# Getting Started with Claude Opus 5 on Amazon Bedrock

**Anthropic's most intelligent model — a step change in coding, agentic autonomy, and professional knowledge work.**

Claude Opus 5 builds on Opus 4.8 with deeper reasoning via adaptive thinking effort levels, new agentic primitives like mid-conversation tool changes, and a simplified API surface. It is available on Amazon Bedrock through **three API paths**, so you can integrate with the Anthropic SDK or boto3.



---

## What's new in Claude Opus 5

- **Mid-conversation changes (beta)** — change the request configuration partway through a conversation, without invalidating the prompt cache:
  - **Tool changes** — add or remove tools per turn
  - **Effort changes** — switch the `output_config.effort` level per turn (e.g. `low` on routine turns, `high` on hard ones)
  - **Task budget changes** — adjust the output token budget per turn for more predictable spend
- **Effort cap when thinking is disabled**: with `thinking: {"type": "disabled"}`, `output_config.effort` is capped at `high`; `xhigh` and `max` require adaptive thinking
- **Frontier capability at Opus-tier economics**: matches Claude Fable 5's top-tier intelligence in many domains at Opus-tier pricing


---

## When to use Claude Opus 5

Claude Opus 5 is a strong fit for industries where precision, reliability, and deep reasoning matter most.

| Use case | Description |
|----------|-------------|
| **Agentic coding** | Understands and navigates codebases like an experienced engineer, writes production-quality code, and adapts its strategy as it works |
| **Long-running agents** | Dependable agents that work for hours or overnight — finding paths around obstacles, recovering from errors, and reaching their objectives with less oversight |
| **Agents & workflow automation** | Pushes back on flawed instructions and breaks complex jobs into sub-agents that need less supervision |
| **Financial services** | Financial workflows with deeper reasoning and end-to-end context, handling compliance-sensitive work with more clarity and focus |
| **Professional & productivity work** | Report building and auditing, document drafting, and structured analysis with high consistency across multi-day projects |
| **Cybersecurity** | Improved cyber capabilities across the board, from coding to security review (higher-risk requests may fall back to Opus 4.8 — see the note below) |

> **Note on fallbacks:** In higher-risk areas, Opus 5 may fall back to Opus 4.8. Users are notified when this happens, and fallback behavior can be configured when using the APIs.

---

## Key capabilities

| Capability | Details |
|------------|---------|
| **Context window** | Up to 1M tokens |
| **Max output** | Up to 128K tokens |
| **Knowledge cutoff** | May 2026 |
| **Adaptive thinking** | On by default; configurable effort levels (`low` through `max`). Can be disabled, but effort is then capped at `high` |
| **Task budgets** | Token spending limits for agentic loops, minimum 20,000 tokens (beta) |
| **Context compaction** | Automatic summarization for effectively infinite context (beta) |
| **Mid-conversation changes** | Switch tools, effort, or task budget mid-conversation without invalidating the prompt cache (beta) |
| **Prompt caching** | Up to 4 cache checkpoints per request, 512-token minimum; `system`, `messages`, and `tools` (5-minute and 1-hour TTL) |
| **Vision** | High-resolution image input for screenshots, diagrams, charts, and dense documents |
| **Computer use** | Strongest Opus-tier model for computer use — pairs high-resolution vision with deep reasoning for multi-step tasks across multiple applications |
| **Data governance** | Zero data retention (ZDR) enabled by default on Amazon Bedrock |


---

## API options on Amazon Bedrock

Claude Opus 5 is available through three API paths:

| # | API | Endpoint | SDK | Auth | Model ID |
|---|-----|----------|-----|------|----------|
| 1 | **Messages API** (recommended) | `bedrock-mantle.{region}.api.aws/anthropic` | `anthropic[bedrock]` | SigV4 | `anthropic.claude-opus-5` |
| 2 | **InvokeModel API** | `bedrock-runtime.{region}.amazonaws.com` | boto3 | SigV4 | `us.anthropic.claude-opus-5` |
| 3 | **Converse API** | `bedrock-runtime.{region}.amazonaws.com` | boto3 | SigV4 | `us.anthropic.claude-opus-5` |


**Model ID notes:**
- Mantle APIs use the bare model ID: `anthropic.claude-opus-5`
- Runtime APIs use a geo-CRIS (cross-region inference) prefix based on your region — e.g. `us.anthropic.claude-opus-5` in US regions or `eu.anthropic.claude-opus-5` in `eu-west-1`. A `global.anthropic.claude-opus-5` profile is also available.



---

## 1. Setup and configuration

In [ ]:
# Install required packages (uncomment if needed)
#!pip install boto3 --upgrade 2>&1 | tail -3
#!pip install "anthropic[bedrock]" --upgrade 2>&1 | tail -3

In [ ]:
import boto3
import json
import base64
from botocore.config import Config

# Try importing Anthropic Bedrock Mantle client
try:
    from anthropic import AnthropicBedrockMantle
    MANTLE_AVAILABLE = True
except ImportError:
    print(
        "Note: anthropic[bedrock] not installed. "
        "Messages API examples will be skipped. "
        "Install with: pip install anthropic[bedrock]"
    )
    MANTLE_AVAILABLE = False

# Configuration
REGION = "us-east-1"

# Mantle (Messages API) - recommended. Bare model ID, no geo prefix.
MANTLE_MODEL_ID = "anthropic.claude-opus-5"
MANTLE_BASE_URL = f"https://bedrock-mantle.{REGION}.api.aws"

# Runtime (Converse / InvokeModel APIs) - geo-CRIS prefixed model ID.
# Use "us." for US/CA/SA regions, "eu." for EU/ME/AF
# A "global." cross-region profile is also available.
RUNTIME_MODEL_ID = "us.anthropic.claude-opus-5"

# Generous timeout: Opus 5 supports up to 128K output tokens,
# and thinking/compaction responses can run long
FAST_CONFIG = Config(read_timeout=1200, retries={"max_attempts": 1})

# Initialize clients
session = boto3.Session()
bedrock_runtime = session.client(
    service_name="bedrock-runtime",
    region_name=REGION,
    config=FAST_CONFIG,
)

if MANTLE_AVAILABLE:
    mantle_client = AnthropicBedrockMantle(
        aws_region=REGION,
        base_url=f"{MANTLE_BASE_URL}/anthropic",
    )

print(f"Region:          {REGION}")
print(f"Mantle Model:    {MANTLE_MODEL_ID}")
print(f"Runtime Model:   {RUNTIME_MODEL_ID}")
print(f"Mantle Base URL: {MANTLE_BASE_URL}")
print(f"Mantle SDK:      {'available' if MANTLE_AVAILABLE else 'not installed'}")
print("Setup complete.")

---

## 2. Basic usage with Messages API (Bedrock Mantle)

The Messages API uses Anthropic's native request/response format, authenticated with AWS SigV4 via the `anthropic[bedrock]` SDK.

In [ ]:
# Messages API - basic request
if MANTLE_AVAILABLE:
    try:
        message = mantle_client.messages.create(
            model=MANTLE_MODEL_ID,
            max_tokens=2048,
            messages=[
                {
                    "role": "user",
                    "content": (
                        "What are three key considerations when designing "
                        "a distributed system for high availability?"
                    ),
                }
            ],
        )

        print(message.content[1].text)
        print(
            f"\nUsage: Input={message.usage.input_tokens}, "
            f"Output={message.usage.output_tokens}"
        )

    except Exception as e:
        print(f"Error: {e}")
else:
    print("Skipping - install anthropic[bedrock] for Messages API support")

---

## 3. Adaptive thinking and effort levels

Adaptive thinking gives Claude the freedom to reason deeply when a task requires it, and respond quickly when it doesn't. You guide reasoning depth with an `effort` parameter rather than fixed token budgets.

**How it works:**
- Set `thinking.type` to `"adaptive"` — Claude decides whether to think based on task complexity
- Control depth with `output_config.effort`
- By default, thinking content is **omitted**: the thinking block contains a cryptographic `signature` proving reasoning occurred, but the text is empty. Set `thinking.display: "summarized"` to receive a summary of the reasoning.

### Effort levels (Opus 5)

| Effort | Behavior | Best for |
|--------|----------|----------|
| **max** | Maximum reasoning depth | The most complex analytical workloads |
| **xhigh** | Above high, below max | Very complex tasks at lower cost than max |
| **high** | Deep reasoning | Complex tasks — start here |
| **medium** | Balanced — may skip thinking for simple queries | Mixed workloads |
| **low** | Minimal thinking, prioritizes speed | Simple queries, cost-sensitive paths |

**Constraints to know:**
- With `thinking: {"type": "disabled"}`, effort is **capped at `high`** — requesting `xhigh` or `max` returns a 400. The cap also applies to any per-turn effort set via a mid-conversation `role: "system"` message.

In [ ]:
# Adaptive thinking - basic example
if MANTLE_AVAILABLE:
    try:
        message = mantle_client.messages.create(
            model=MANTLE_MODEL_ID,
            max_tokens=16000,
            thinking={"type": "adaptive"},
            extra_body={"output_config": {"effort": "high"}},
            messages=[
                {
                    "role": "user",
                    "content": (
                        "Design a fault-tolerant distributed consensus algorithm "
                        "that handles Byzantine failures in an asynchronous "
                        "network with up to 1/3 faulty nodes."
                    ),
                }
            ],
        )

        has_thinking = any(block.type == "thinking" for block in message.content)
        print(f"Claude decided to think: {has_thinking}\n")

        for block in message.content:
            if block.type == "thinking":
                sig_len = len(block.signature) if block.signature else 0
                print(f"[Thinking block] Signature: {sig_len} chars")
                print("(Thinking omitted by default - signature proves reasoning occurred)\n")
            elif block.type == "text":
                text = block.text
                print(f"Response:\n{text[:1000]}..." if len(text) > 1000 else f"Response:\n{text}")

        print(
            f"\nUsage: Input={message.usage.input_tokens}, "
            f"Output={message.usage.output_tokens}"
        )

    except Exception as e:
        print(f"Error: {e}")
else:
    print("Skipping - install anthropic[bedrock]")

In [ ]:
# Compare all five effort levels on the same prompt
def test_effort_level(effort_level: str, prompt: str):
    if not MANTLE_AVAILABLE:
        return {"effort": effort_level, "error": "SDK not installed"}

    try:
        message = mantle_client.messages.create(
            model=MANTLE_MODEL_ID,
            max_tokens=12000,
            thinking={"type": "adaptive"},
            extra_body={"output_config": {"effort": effort_level}},
            messages=[{"role": "user", "content": prompt}],
        )

        has_thinking = any(b.type == "thinking" for b in message.content)
        response_len = sum(len(b.text) for b in message.content if b.type == "text")

        return {
            "effort": effort_level,
            "thought": has_thinking,
            "input_tokens": message.usage.input_tokens,
            "output_tokens": message.usage.output_tokens,
            "response_chars": response_len,
        }
    except Exception as e:
        return {"effort": effort_level, "error": str(e)}


prompt = (
    "Prove that there are infinitely many prime numbers. "
    "Then explain why this proof technique cannot be used to prove "
    "the twin prime conjecture."
)

print("Testing effort levels on the same prompt...\n")
print(f"{'Effort':<8} {'Thought':<8} {'Input':<8} {'Output':<8} {'Response':<10}")
print("-" * 50)

for effort in ["low", "medium", "high", "xhigh", "max"]:
    result = test_effort_level(effort, prompt)
    if "error" not in result:
        print(
            f"{result['effort']:<8} "
            f"{str(result['thought']):<8} "
            f"{result['input_tokens']:<8} "
            f"{result['output_tokens']:<8} "
            f"{result['response_chars']:<10}"
        )
    else:
        print(f"{result['effort']:<8} Error: {result['error'][:60]}")

print("\nHigher effort = deeper reasoning, more tokens, better quality on complex tasks.")

In [ ]:
# Thinking display modes: omitted (default) vs summarized
if MANTLE_AVAILABLE:
    try:
        # Summarized: receive a readable summary of the reasoning
        message = mantle_client.messages.create(
            model=MANTLE_MODEL_ID,
            max_tokens=4000,
            thinking={"type": "adaptive", "display": "summarized"},
            extra_body={"output_config": {"effort": "max"}},
            messages=[{"role": "user", "content": "What is 27 * 453? Show your work."}],
        )

        for block in message.content:
            if block.type == "thinking":
                print(f"[Summarized thinking] ({len(block.thinking)} chars):")
                print(block.thinking[:500])
                print()
            elif block.type == "text":
                print(f"Answer: {block.text}")

    except Exception as e:
        print(f"Error: {e}")
else:
    print("Skipping - install anthropic[bedrock]")

In [ ]:
# Effort cap when thinking is disabled:
# effort 'high' (or below) is fine, 'xhigh'/'max' return a 400
if MANTLE_AVAILABLE:
    # Accepted: thinking disabled + effort=high
    try:
        message = mantle_client.messages.create(
            model=MANTLE_MODEL_ID,
            max_tokens=1000,
            thinking={"type": "disabled"},
            extra_body={"output_config": {"effort": "high"}},
            messages=[{"role": "user", "content": "What is 27 * 453?"}],
        )
        print(f"thinking=disabled + effort=high accepted: {message.content[0].text[:60]}\n")
    except Exception as e:
        print(f"Error: {e}\n")

    # Rejected: thinking disabled + effort=xhigh
    try:
        mantle_client.messages.create(
            model=MANTLE_MODEL_ID,
            max_tokens=1000,
            thinking={"type": "disabled"},
            extra_body={"output_config": {"effort": "xhigh"}},
            messages=[{"role": "user", "content": "What is 2+2?"}],
        )
        print("Unexpected: xhigh accepted with thinking disabled")
    except Exception as e:
        print(f"thinking=disabled + effort=xhigh correctly rejected:\n  {str(e)[:150]}")
else:
    print("Skipping - install anthropic[bedrock]")

---

## 5. Structured outputs (JSON schema)

Structured outputs (`response_format: json_schema`) is not currently supported on Amazon Bedrock — requests using it will return a 400 error. This is a known feature gap.

**Workarounds:**

- **JSON via prompting** — request JSON output with an example shape in the prompt. Opus 5 follows this reliably.
- **Output format via literal example** — include a concrete example of the desired format (e.g., "a coordinate pair (x, y), e.g. (123, 456)"). A single example is often sufficient.

The cell below demonstrates the JSON-via-prompting workaround:

In [ ]:
# JSON via prompting (structured-output fallback) — include a literal example
if MANTLE_AVAILABLE:
    try:
        message = mantle_client.messages.create(
            model=MANTLE_MODEL_ID,
            max_tokens=256,
            messages=[
                {
                    "role": "user",
                    "content": (
                        "List exactly 3 programming languages with their year of "
                        "creation. Respond with ONLY a JSON object, no prose and no "
                        "markdown code fences, matching this shape exactly:\n"
                        '{"languages": [{"name": "Python", "year": 1991}]}'
                    ),
                }
            ],
        )

        text = message.content[0].text.strip()
        # Tolerate an accidental code fence
        if text.startswith("```"):
            text = text.strip("`")
            text = text[text.find("{"):text.rfind("}") + 1]
        parsed = json.loads(text)
        print(json.dumps(parsed, indent=2))

    except Exception as e:
        print(f"Error: {e}")
else:
    print("Skipping - install anthropic[bedrock]")

---

## 6. Task budgets (beta)

Task budgets let you set token spending limits for entire agentic loops. This prevents runaway costs during autonomous operations while allowing Claude to complete complex multi-step tasks.

**How it works:**
- Set a maximum token budget via `output_config.task_budget`
- Claude tracks usage across the conversation/agentic loop
- When the budget is approached, Claude wraps up gracefully rather than stopping mid-task
- **Minimum budget: 20,000 tokens** — lower values return a 400 error
- Combines with adaptive thinking and effort levels

**Requires beta header:** `anthropic_beta: ["task-budgets-2026-03-13"]`

In [ ]:
# Task budget example (combined with adaptive thinking + effort)
if MANTLE_AVAILABLE:
    try:
        message = mantle_client.messages.create(
            model=MANTLE_MODEL_ID,
            max_tokens=8000,
            thinking={"type": "adaptive"},
            extra_body={
                "anthropic_beta": ["task-budgets-2026-03-13"],
                "output_config": {
                    "effort": "high",
                    "task_budget": {"type": "tokens", "total": 25000},
                },
            },
            messages=[
                {
                    "role": "user",
                    "content": (
                        "Write a Python implementation of a binary search tree "
                        "with insert, delete, and search methods. Include docstrings."
                    ),
                }
            ],
        )

        print(f"Stop reason: {message.stop_reason}")
        print(f"Output tokens used: {message.usage.output_tokens} (budget: 25,000)\n")

        for block in message.content:
            if block.type == "text":
                text = block.text
                print(f"{text[:1000]}..." if len(text) > 1000 else text)

    except Exception as e:
        print(f"Error: {e}")
        print("Note: Task budgets require the beta header and a minimum budget of 20,000 tokens.")
else:
    print("Skipping - install anthropic[bedrock]")

---

## 7. Context compaction (beta)

Context compaction enables "infinite context" for long-running conversations and agentic tasks by automatically summarizing older context when approaching the context window limit.

**How it works:**
1. Enable compaction with `context_management.edits` containing a `compact_20260112` edit
2. Set a `trigger` threshold — when input exceeds this, Claude summarizes older context
3. The API returns a compaction content block in the response
4. Pass the compaction block back in subsequent requests to maintain continuity

**Available with:** Messages API and InvokeModel API (**not** Converse API)

**Requires beta header:** `anthropic_beta: ["compact-2026-01-12"]`

In [ ]:
# Context compaction - basic setup
if MANTLE_AVAILABLE:
    try:
        message = mantle_client.messages.create(
            model=MANTLE_MODEL_ID,
            max_tokens=1024,
            extra_body={
                "anthropic_beta": ["compact-2026-01-12"],
                "context_management": {
                    "edits": [
                        {
                            "type": "compact_20260112",
                            "trigger": {
                                "type": "input_tokens",
                                "value": 100000,  # Trigger compaction at 100K tokens
                            },
                        }
                    ]
                },
            },
            messages=[
                {
                    "role": "user",
                    "content": (
                        "Let's start a long-running conversation about building "
                        "a complex e-commerce platform. What are the key "
                        "architectural decisions we need to make first?"
                    ),
                }
            ],
        )

        has_compaction = any(
            getattr(b, "type", None) == "compaction" for b in message.content
        )
        print(f"Compaction triggered: {has_compaction}")
        print("(Won't trigger on short conversations - designed for 100K+ token contexts)\n")

        for block in message.content:
            if block.type == "text":
                text = block.text
                print(f"{text[:800]}..." if len(text) > 800 else text)

    except Exception as e:
        print(f"Error: {e}")
        print("Note: Compaction requires the beta header.")
else:
    print("Skipping - install anthropic[bedrock]")

In [ ]:
# Multi-turn conversation with automatic compaction handling
if MANTLE_AVAILABLE:
    conversation = []

    def chat_with_compaction(user_message: str):
        """Send a message and handle compaction blocks automatically."""
        conversation.append({"role": "user", "content": user_message})

        message = mantle_client.messages.create(
            model=MANTLE_MODEL_ID,
            max_tokens=4096,
            messages=conversation,
            extra_body={
                "anthropic_beta": ["compact-2026-01-12"],
                "context_management": {
                    "edits": [
                        {
                            "type": "compact_20260112",
                            "trigger": {"type": "input_tokens", "value": 100000},
                        }
                    ]
                },
            },
        )

        # Build assistant response preserving compaction blocks
        assistant_content = []
        response_text = ""
        compacted = False

        for block in message.content:
            if block.type == "text":
                assistant_content.append({"type": "text", "text": block.text})
                response_text += block.text
            elif block.type == "compaction":
                assistant_content.append(
                    {"type": "compaction", "compaction": block.compaction}
                )
                compacted = True

        conversation.append({"role": "assistant", "content": assistant_content})

        if compacted:
            print("[Compaction occurred - older context was summarized]\n")

        return response_text

    try:
        print("Turn 1:")
        r1 = chat_with_compaction("Design the database schema for an e-commerce platform")
        print(f"{r1[:400]}...\n")
        print("=" * 50)

        print("\nTurn 2:")
        r2 = chat_with_compaction("Now add authentication and authorization")
        print(f"{r2[:400]}...\n")

        print(f"Conversation: {len(conversation)} messages")

    except Exception as e:
        print(f"Error: {e}")
else:
    print("Skipping - install anthropic[bedrock]")

---

## 8. Tool use, prompt caching, and mid-conversation tool changes

Claude Opus 5 excels at multi-step tool use. This section covers:

1. **Client-defined tools** — standard tool use
2. **Prompt caching** — cache large system prompts/tool definitions with `cache_control`
3. **Mid-conversation tool changes (beta, new in Opus 5)** — add or remove tools mid-conversation *without invalidating the prompt cache*

In [ ]:
# Client-defined tool use
if MANTLE_AVAILABLE:
    tools = [
        {
            "name": "calculator",
            "description": "Performs basic arithmetic. Use for any math question.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "The math expression to evaluate",
                    }
                },
                "required": ["expression"],
            },
        }
    ]

    try:
        message = mantle_client.messages.create(
            model=MANTLE_MODEL_ID,
            max_tokens=512,
            tools=tools,
            messages=[
                {"role": "user", "content": "What is 42 * 17? Use the calculator tool."}
            ],
        )

        print(f"Stop reason: {message.stop_reason}")
        for block in message.content:
            if block.type == "tool_use":
                print(f"Tool call: {block.name}({json.dumps(block.input)})")
                # In production: execute the tool, then send a tool_result
                # message back and continue the loop until stop_reason='end_turn'
            elif block.type == "text":
                print(f"Text: {block.text[:200]}")

    except Exception as e:
        print(f"Error: {e}")
else:
    print("Skipping - install anthropic[bedrock]")

In [ ]:
# Prompt caching - cache a large system prompt with cache_control
if MANTLE_AVAILABLE:
    large_system = "You are a helpful coding assistant.\n" + "\n".join(
        [
            f"Rule {i}: Always write clean, well-documented, production-ready code. "
            f"Follow best practices for error handling, testing, and performance."
            for i in range(1, 251)
        ]
    )

    kwargs = dict(
        model=MANTLE_MODEL_ID,
        max_tokens=64,
        system=[
            {
                "type": "text",
                "text": large_system,
                "cache_control": {"type": "ephemeral"},
            }
        ],
        messages=[{"role": "user", "content": "Say hello briefly."}],
    )

    try:
        # First request populates the cache
        msg1 = mantle_client.messages.create(**kwargs)
        cache_write = getattr(msg1.usage, "cache_creation_input_tokens", 0) or 0

        # Second request should read from the cache
        msg2 = mantle_client.messages.create(**kwargs)
        cache_read = getattr(msg2.usage, "cache_read_input_tokens", 0) or 0

        print(f"Request 1 - cache write: {cache_write} tokens")
        print(f"Request 2 - cache read:  {cache_read} tokens")
        print("\nCached tokens are billed at a significant discount vs. regular input tokens.")

    except Exception as e:
        print(f"Error: {e}")
else:
    print("Skipping - install anthropic[bedrock]")

### Mid-conversation changes (beta) — new in Opus 5

Opus 5 lets you change the request configuration partway through a conversation — via `role: "system"` messages — **without invalidating the prompt cache**. Three per-turn changes launch together:

- **Tool changes** — add or remove tools mid-conversation (demonstrated below)
- **Effort changes** — switch the `output_config.effort` level per turn (e.g. `low` on routine turns, `high` on hard ones)
- **Task budget changes** — adjust the output token budget per turn for more predictable spend

Previously, changing any of these mid-conversation meant re-sending the full top-level request — which invalidated the prompt cache. The per-turn mechanism preserves it.

> **Runnable example below covers tool changes.** Per-turn **effort** and **task-budget** changes share the same cache-preserving mechanism; their exact request shapes are still being confirmed for this notebook, so they're described here rather than demonstrated.

### Tool changes — rules

- Requires beta header: `anthropic_beta: ["mid-conversation-tool-changes-2026-07-01"]`
- `tool_addition` / `tool_removal` blocks are only permitted inside `role: "system"` messages (placing them elsewhere returns a 400)

### Tool reference variants

Each `tool_addition` / `tool_removal` block carries a `tool` field — a discriminated union on `type`:

| `type` | Fields | Refers to | Extra beta |
|--------|--------|-----------|------------|
| `tool_reference` | `name` (pattern `^[a-zA-Z0-9_-]{1,128}$`) | A tool declared directly in top-level `tools[]` | — |
| `mcp_tool_reference` | `server_name`, `name` | A single MCP tool | an `mcp-client-*` beta |
| `mcp_toolset_reference` | `server_name` | Every tool in the named MCP server's toolset | an `mcp-client-*` beta |

`tool_reference` does **not** accept the composed `{server}_{name}` form assigned to MCP-resolved tools — use one of the MCP variants for those. A `tool_reference` naming a tool not in `tools[]` returns a 400.

### Resolution semantics and limits

- The available-tool set starts as everything in `tools[]` (after MCP resolution). Each `tool_removal` subtracts the referenced tool(s); each `tool_addition` re-adds them. A tool removed and later re-added is available at the end.
- `tool_removal` renders a brief in-context notice so the model stops planning around the removed tool.
- Mid-conversation tool changes do **not** trigger constrained decoding — `tool_choice` remains the only control for that.
- Maximum **512** `tool_addition` blocks per request.

In [ ]:
# Mid-conversation tool changes (beta)
MID_CONV_TOOL_CHANGES_BETA = "mid-conversation-tool-changes-2026-07-01"

if MANTLE_AVAILABLE:
    tools = [
        {
            "name": "get_weather",
            "description": "Get the current weather for a location.",
            "input_schema": {
                "type": "object",
                "properties": {"location": {"type": "string"}},
                "required": ["location"],
            },
        },
        {
            "name": "search_docs",
            "description": "Search internal documentation.",
            "input_schema": {
                "type": "object",
                "properties": {"query": {"type": "string"}},
                "required": ["query"],
            },
        },
    ]

    try:
        message = mantle_client.messages.create(
            model=MANTLE_MODEL_ID,
            max_tokens=512,
            tools=tools,
            extra_body={"anthropic_beta": [MID_CONV_TOOL_CHANGES_BETA]},
            messages=[
                {"role": "user", "content": "Look up the weather in Seattle."},
                # Phase change: remove weather tool, add docs search tool.
                # This preserves the prompt cache, unlike re-sending tools[].
                {
                    "role": "system",
                    "content": [
                        {
                            "type": "tool_removal",
                            "tool": {"type": "tool_reference", "name": "get_weather"},
                        },
                        {
                            "type": "tool_addition",
                            "tool": {"type": "tool_reference", "name": "search_docs"},
                        },
                        {"type": "text", "text": "Tool set updated for phase 2."},
                    ],
                },
                {"role": "user", "content": "Now search the docs for 'prompt caching'."},
            ],
        )

        tool_names = [b.name for b in message.content if b.type == "tool_use"]
        print(f"Stop reason: {message.stop_reason}")
        print(f"Tools called: {tool_names}")
        print("(Should only use search_docs - get_weather was removed mid-conversation)")

    except Exception as e:
        print(f"Error: {e}")
        print("Note: requires the mid-conversation-tool-changes beta header.")
else:
    print("Skipping - install anthropic[bedrock]")

---

## 9. Vision

Claude Opus 5 supports high-resolution image input for screenshots, charts, dense documents, and complex diagrams — useful for financial analysis (dense filings and charts) and cybersecurity (screenshots and logs).

In [ ]:
# Vision example - generate a small sample chart PNG, then analyze it
import struct, zlib

def create_sample_chart_png():
    """Create a simple 200x100 PNG with colored bars (simulates a bar chart)."""
    w, h = 200, 100
    bar_colors = [(66, 133, 244), (234, 67, 53), (251, 188, 4), (52, 168, 83)]
    bar_heights = [80, 55, 70, 90]
    bar_width, gap, start_x = 30, 15, 20
    raw = b''
    for y in range(h):
        raw += b'\x00'
        for x in range(w):
            drawn = False
            for i, (bh, color) in enumerate(zip(bar_heights, bar_colors)):
                bx = start_x + i * (bar_width + gap)
                if bx <= x < bx + bar_width and y >= h - bh:
                    raw += bytes(color)
                    drawn = True
                    break
            if not drawn:
                raw += b'\xff\xff\xff'
    def chunk(ct, d):
        c = ct + d
        return struct.pack('>I', len(d)) + c + struct.pack('>I', zlib.crc32(c) & 0xFFFFFFFF)
    ihdr = struct.pack('>IIBBBBB', w, h, 8, 2, 0, 0, 0)
    return (b'\x89PNG\r\n\x1a\n'
            + chunk(b'IHDR', ihdr)
            + chunk(b'IDAT', zlib.compress(raw))
            + chunk(b'IEND', b''))

sample_image = create_sample_chart_png()
image_b64 = base64.b64encode(sample_image).decode('utf-8')
print(f"Generated sample chart: {len(sample_image)} bytes")
print("(In production, use real high-res images to leverage Opus 5's vision)\n")

if MANTLE_AVAILABLE:
    try:
        message = mantle_client.messages.create(
            model=MANTLE_MODEL_ID,
            max_tokens=2048,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "image",
                            "source": {
                                "type": "base64",
                                "media_type": "image/png",
                                "data": image_b64,
                            },
                        },
                        {
                            "type": "text",
                            "text": "Describe this chart. What type is it? What do the colors represent?",
                        },
                    ],
                }
            ],
        )

        print(f"Vision Analysis:\n{message.content[0].text}")

    except Exception as e:
        print(f"Error: {e}")
else:
    print("Skipping - install anthropic[bedrock]")

---

## 10. Bedrock Runtime APIs (Converse and InvokeModel)

For teams already using Bedrock's native APIs, Claude Opus 5 works with both Converse and InvokeModel via boto3. Use the **geo-CRIS prefixed model ID** (e.g. `us.anthropic.claude-opus-5`).

- **Converse**: Bedrock's unified conversational API. Pass Opus 5 features (adaptive thinking, effort, structured output) via `additionalModelRequestFields`. Compaction is **not** supported on Converse.
- **InvokeModel**: Anthropic request body format. Supports compaction, task budgets, structured output — `thinking` and `output_config` go directly in the request body.

In [ ]:
# Converse API - basic + adaptive thinking via additionalModelRequestFields
try:
    response = bedrock_runtime.converse(
        modelId=RUNTIME_MODEL_ID,
        messages=[
            {
                "role": "user",
                "content": [{"text": "What is the sum of the first 20 prime numbers? Show your work."}],
            }
        ],
        inferenceConfig={"maxTokens": 8000},
        additionalModelRequestFields={
            "thinking": {"type": "adaptive"},
            "output_config": {"effort": "high"},
        },
    )

    content = response["output"]["message"]["content"]
    has_thinking = any(block.get("type") == "thinking" for block in content)
    print(f"Thinking occurred: {has_thinking}\n")

    for block in content:
        if block.get("text"):
            text = block["text"]
            print(f"{text[:600]}..." if len(text) > 600 else text)

    print(
        f"\nUsage: Input={response['usage']['inputTokens']}, "
        f"Output={response['usage']['outputTokens']}"
    )

except Exception as e:
    print(f"Error: {e}")

In [ ]:
# Converse API - tool use with Bedrock's toolConfig format
try:
    tool_config = {
        "tools": [
            {
                "toolSpec": {
                    "name": "calculator",
                    "description": "Performs basic arithmetic.",
                    "inputSchema": {
                        "json": {
                            "type": "object",
                            "properties": {
                                "expression": {
                                    "type": "string",
                                    "description": "Math expression",
                                }
                            },
                            "required": ["expression"],
                        }
                    },
                }
            }
        ]
    }

    response = bedrock_runtime.converse(
        modelId=RUNTIME_MODEL_ID,
        messages=[
            {
                "role": "user",
                "content": [{"text": "What is 42 * 17? Use the calculator tool."}],
            }
        ],
        inferenceConfig={"maxTokens": 512},
        toolConfig=tool_config,
    )

    print(f"Stop reason: {response.get('stopReason')}")
    for block in response["output"]["message"]["content"]:
        if block.get("toolUse"):
            tu = block["toolUse"]
            print(f"Tool call: {tu['name']}({json.dumps(tu['input'])})")
        elif block.get("text"):
            print(f"Text: {block['text'][:200]}")

except Exception as e:
    print(f"Error: {e}")

In [ ]:
# InvokeModel API - adaptive thinking + effort in the request body
request_body = {
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 8000,
    "thinking": {"type": "adaptive"},
    "output_config": {"effort": "high"},
    "messages": [
        {
            "role": "user",
            "content": "Explain the CAP theorem and its practical implications for database design.",
        }
    ],
}

try:
    response = bedrock_runtime.invoke_model(
        modelId=RUNTIME_MODEL_ID, body=json.dumps(request_body)
    )
    body = json.loads(response["body"].read())

    has_thinking = any(b["type"] == "thinking" for b in body["content"])
    print(f"Thinking occurred: {has_thinking}\n")

    for block in body["content"]:
        if block["type"] == "text":
            text = block["text"]
            print(f"{text[:600]}..." if len(text) > 600 else text)

    print(
        f"\nUsage: Input={body['usage']['input_tokens']}, "
        f"Output={body['usage']['output_tokens']}"
    )

except Exception as e:
    print(f"Error: {e}")

In [ ]:
# InvokeModel API - streaming
request_body = {
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 512,
    "messages": [
        {"role": "user", "content": "Count from 1 to 5, one per line."}
    ],
}

try:
    response = bedrock_runtime.invoke_model_with_response_stream(
        modelId=RUNTIME_MODEL_ID, body=json.dumps(request_body)
    )
    for event in response["body"]:
        chunk = json.loads(event["chunk"]["bytes"])
        if chunk.get("type") == "content_block_delta":
            delta = chunk.get("delta", {})
            if delta.get("type") == "text_delta":
                print(delta.get("text", ""), end="", flush=True)
    print()

except Exception as e:
    print(f"Error: {e}")

---

## 11. Fallbacks to Opus 4.8

Claude Opus 5 can be used to scan source code for vulnerabilities, triage security issues, and build secure code. In higher-risk areas — such as exploit generation, binary-based vulnerability scanning, and penetration testing — it may decline a request and return `stop_reason: "refusal"` instead of a normal completion, falling back to Claude Opus 4.8. To keep your application responsive, retry the same request on the fallback model.

The pattern below is a simple client-side retry: send to Opus 5, check `stop_reason`, and re-send to the fallback model on refusal. It uses only standard Messages API calls, so it works unchanged on Bedrock.

> **Note:** A server-side fallback option (a single request that fails over automatically) and a "fallback credit" feature that avoids re-paying cache costs on the retry also exist as betas. Their request/response shapes are still being confirmed for Bedrock — this notebook uses the portable client-side pattern instead.

In [ ]:
# Client-side fallback: retry on Opus 4.8 when Opus 5 refuses
def invoke_with_fallback(messages, system=None, tools=None,
                         primary_model=MANTLE_MODEL_ID,
                         fallback_model="anthropic.claude-opus-4-8",
                         max_tokens=4096):
    """Send to the primary model; on stop_reason='refusal', retry on the fallback model."""
    if not MANTLE_AVAILABLE:
        print("Skipping - install anthropic[bedrock]")
        return None

    kwargs = {"max_tokens": max_tokens, "messages": messages}
    if system:
        kwargs["system"] = system
    if tools:
        kwargs["tools"] = tools

    try:
        message = mantle_client.messages.create(model=primary_model, **kwargs)

        if message.stop_reason == "refusal":
            print(f"{primary_model} refused the request. Falling back to {fallback_model}...\n")
            fallback = mantle_client.messages.create(model=fallback_model, **kwargs)
            if fallback.stop_reason == "refusal":
                print(f"Fallback model {fallback_model} also refused.")
                return None
            print(f"Fallback succeeded on {fallback_model}.")
            return next((b.text for b in fallback.content if b.type == "text"), None)

        print(f"Primary model {primary_model} responded successfully.")
        return next((b.text for b in message.content if b.type == "text"), None)

    except Exception as e:
        print(f"Error: {e}")
        return None


# Example
result = invoke_with_fallback(
    [{"role": "user", "content": "Summarize the key considerations for securing a REST API."}]
)
if result:
    print(f"\n{result[:400]}")

---

## 12. Next steps

- **Migrate from earlier Claude models**: remove `temperature`/`top_p`/`top_k` and any `budget_tokens` thinking config — switch to adaptive thinking with effort levels
- **Calibrate effort** on your workloads: start at `high`, step up to `xhigh`/`max` for the hardest tasks, drop to `medium`/`low` for cost-sensitive paths
- **Enable compaction** for long-running agents to avoid context window limits
- **Set task budgets** (≥20K tokens) for autonomous workflows to control costs
- **Use mid-conversation tool changes** to phase tool sets in multi-stage agents while preserving your prompt cache
- **Reuse existing Anthropic SDK code** via the Messages API on mantle endpoint

### Model ID quick reference

| Context | Model ID |
|---------|----------|
| Mantle APIs (Messages) | `anthropic.claude-opus-5` |
| Runtime APIs, US/CA/SA regions | `us.anthropic.claude-opus-5` |
| Runtime APIs, EU/ME/AF regions | `eu.anthropic.claude-opus-5` |
| Global cross-region profile | `global.anthropic.claude-opus-5` |

### Beta headers quick reference

| Feature | Beta header | APIs |
|---------|-------------|------|
| Context compaction | `compact-2026-01-12` | Messages, InvokeModel |
| Task budgets | `task-budgets-2026-03-13` | Messages, InvokeModel |
| Mid-conversation tool changes | `mid-conversation-tool-changes-2026-07-01` | Messages |

### Resources

- [Amazon Bedrock documentation](https://docs.aws.amazon.com/bedrock/latest/userguide/)
- [Anthropic SDK for Bedrock Mantle](https://docs.anthropic.com/en/api/claude-on-amazon-bedrock)
- [Claude model documentation](https://docs.anthropic.com/en/docs/about-claude/models)